In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib as mpl
from scipy.stats import pearsonr
import os
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML
import base64
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")
import seaborn as sns

# RPy2 imports
import rpy2.robjects as ro
import rpy2.robjects.packages as rpackages
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Force R to search Conda environment library path
conda_prefix = os.environ.get('CONDA_PREFIX', '/srv/conda/envs/notebook')
r_lib_path = os.path.join(conda_prefix, 'lib', 'R', 'library')
ro.r(f'.libPaths(unique(c("{r_lib_path}", .libPaths())))')

afex = importr('afex')
emmeans = importr('emmeans')
stats = importr('stats')
base = importr('base')

# --- 1. DATA PREPARATION ---
def get_data_path(filename):
    if os.path.exists(os.path.join('data', filename)):
        return os.path.join('data', filename)
    elif os.path.exists(os.path.join('..', 'data', filename)):
        return os.path.join('..', 'data', filename)
    else:
        raise FileNotFoundError(f"Could not find {filename}")

try:
    d1 = pd.read_excel(get_data_path('df1_EVs.xlsx'))
    d2 = pd.read_excel(get_data_path('df2_EVs.xlsx'))
    d3 = pd.read_excel(get_data_path('df3_EVs.xlsx'))
    d4 = pd.read_excel(get_data_path('df4_EVs.xlsx'))
    
    for d in [d1, d2, d3, d4]:
        if 'time' in d.columns:
            d['time'] = d['time'].map({
                'time1': 'baseline', 'time2': '3min', 'time3': '1hr', 'time4': '2hrs',
                'Baseline': 'baseline', '3 min': '3min', '1 hour': '1hr', '2 hours': '2hrs'
            })
    
    long_df = pd.concat([d1, d2, d3, d4], ignore_index=True)
except Exception as e:
    print(f"⚠️ Data loading warning: {e}")

# Standardize subject ID column
possible_id_cols = ['Subject_ID', 'ID', 'id', 'Subject', 'subject', 'Participant', 'Participant_ID', 'ID_Code']
subject_col = next((col for col in possible_id_cols if col in long_df.columns), 'Subject_ID')

#print(f"✅ Identified Subject ID Column: '{subject_col}'")

if subject_col is None:
    # Fallback: create synthetic ID per subject
    long_df['ID'] = long_df.groupby('sex').cumcount() + 1
    subject_col = 'ID'

# Clean column names
long_df[subject_col] = long_df[subject_col].astype(str)
if 'sex' in long_df.columns:
    long_df['sex'] = long_df['sex'].astype(str).str.strip().str.title()

# --- 2. BRACKET ANNOTATION UTILITY ---
def add_bracket(ax, x1, x2, y, h, text):
    """Draws a significance bracket between two x coordinates."""
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.2, c='black')
    ax.text((x1+x2)*0.5, y+h, text, ha='center', va='bottom', color='black', fontsize=7, fontweight='bold')


# --- 3. FULL RM-ANCOVA & POST-HOC ENGINE (JAMOVI MATCHED) ---
def run_full_ancova_with_posthoc(
    df_input,
    subject_col="Subject_ID",
    time_col="time",
    group_col="sex",
    dv_cols=["Concentration/ml", "Median Value (nm)"],
    adjust_method="holm",  # "holm", "fdr", or "tukey"
    alpha=0.05
):
    df = df_input.copy()
    
    if subject_col not in df.columns:
        possible_ids = ['Subject_ID', 'ID', 'id', 'Subject', 'subject', 'Participant']
        subject_col = next((col for col in possible_ids if col in df.columns), subject_col)
    
    def clean_id(val):
        return str(val).strip().replace('.0', '')

    df[subject_col] = df[subject_col].apply(clean_id)
    df[group_col] = df[group_col].astype(str).str.strip().str.title()
    df[time_col] = df[time_col].astype(str).str.strip().str.lower()
    
    time_map = {
        'time1': 'baseline', 'time 1': 'baseline', '1': 'baseline', 'baseline': 'baseline',
        'time2': '3min', 'time 2': '3min', '2': '3min', '3min': '3min', '3 min': '3min',
        'time3': '1hr', 'time 3': '1hr', '3': '1hr', '1hr': '1hr', '1 hour': '1hr',
        'time4': '2hrs', 'time 4': '2hrs', '4': '2hrs', '2hrs': '2hrs', '2 hours': '2hrs'
    }
    df[time_col] = df[time_col].map(lambda x: time_map.get(x, x))
    
    ancova_results = []
    posthoc_results = []
    post_times = ["3min", "1hr", "2hrs"]
    
    for dv in dv_cols:
        if dv not in df.columns:
            continue
            
        sub = df.dropna(subset=[dv, group_col, time_col, subject_col]).copy()
        sub[dv] = pd.to_numeric(sub[dv], errors='coerce')
        sub = sub.dropna(subset=[dv])

        baseline_df = sub[sub[time_col] == 'baseline'][[subject_col, dv]].copy()
        if baseline_df.empty:
            continue
            
        baseline_df = baseline_df.groupby(subject_col)[dv].mean().reset_index()
        baseline_df = baseline_df.rename(columns={dv: 'BaselineValue'})

        post_df = sub[sub[time_col].isin(post_times)].copy()
        if post_df.empty:
            continue

        dat = post_df.merge(baseline_df, on=subject_col, how='inner')
        dat = dat.rename(columns={dv: 'DV', subject_col: 'Subject_ID', group_col: 'Group', time_col: 'TimePoint'})
        dat = dat[['Subject_ID', 'Group', 'TimePoint', 'BaselineValue', 'DV']].dropna()

        if len(dat) < 6:
            continue

        with localconverter(ro.default_converter + pandas2ri.converter):
            r_df = ro.conversion.py2rpy(dat)
            ro.globalenv['d'] = r_df

        ro.r('''
        d$TimePoint <- factor(d$TimePoint, levels=c("3min", "1hr", "2hrs"))
        d$Group <- factor(d$Group)
        d$Subject_ID <- factor(d$Subject_ID)
        d$BaselineValue <- as.numeric(as.character(d$BaselineValue))
        d$DV <- as.numeric(as.character(d$DV))
        d <- na.omit(d)
        ''')

        try:
            # 1. FIT RM-ANCOVA MODEL
            ro.r('''
            suppressMessages({
                a <- afex::aov_ez(
                    id = "Subject_ID",
                    dv = "DV",
                    data = d,
                    within = "TimePoint",
                    between = "Group",
                    covariates = "BaselineValue",
                    type = 3,
                    anova_table = list(correction = "GG", es = "pes")
                )
                tab <- as.data.frame(a$anova_table)
                tab$Effect <- rownames(tab)
            })
            ''')

            with localconverter(ro.default_converter + pandas2ri.converter):
                anova_table = ro.conversion.rpy2py(ro.r('tab'))
                
            n_subjects = dat['Subject_ID'].nunique()
            for idx, row in anova_table.iterrows():
                ancova_results.append({
                    'Variable': dv,
                    'Effect': row.get('Effect', idx),
                    'N': n_subjects,
                    'num_df': row.get('num Df', np.nan),
                    'den_df': row.get('den Df', np.nan),
                    'F_statistic': row.get('F', np.nan),
                    'p_value_raw': row.get('Pr(>F)', np.nan),
                    'Partial_Eta_Squared': row.get('pes', np.nan)
                })

            # 2. RUN POST-HOC (GET BOTH UNADJUSTED AND ADJUSTED P-VALUES)
            ro.r(f'''
            suppressMessages({{
                emm_bg <- emmeans::emmeans(a, specs = ~ Group | TimePoint)
                bg_none <- as.data.frame(pairs(emm_bg, adjust = "none"))
                bg_adj <- as.data.frame(pairs(emm_bg, adjust = "{adjust_method}"))
                bg_none$p.value_adj <- bg_adj$p.value
                
                emm_wg <- emmeans::emmeans(a, specs = ~ TimePoint | Group)
                wg_none <- as.data.frame(pairs(emm_wg, adjust = "none"))
                wg_adj <- as.data.frame(pairs(emm_wg, adjust = "{adjust_method}"))
                wg_none$p.value_adj <- wg_adj$p.value
            }})
            ''')

            with localconverter(ro.default_converter + pandas2ri.converter):
                bg_df = ro.conversion.rpy2py(ro.r('bg_none'))
                wg_df = ro.conversion.rpy2py(ro.r('wg_none'))
                
            bg_df['Comparison_Type'] = 'Between-Group (Male vs Female)'
            bg_df['Variable'] = dv
            
            wg_df['Comparison_Type'] = 'Within-Group (Time Point Changes)'
            wg_df['Variable'] = dv
            
            posthoc_results.extend([bg_df, wg_df])

        except Exception as e:
            print(f"Error computing model for {dv}: {e}")
            continue

    # Main ANCOVA Table Processing
    ancova_df = pd.DataFrame(ancova_results)
    if not ancova_df.empty:
        valid_p = ancova_df['p_value_raw'].dropna()
        if not valid_p.empty:
            # Multi-analyte panel FDR adjustment across main effects
            _, p_adj, _, _ = multipletests(valid_p, alpha=alpha, method='fdr_bh')
            ancova_df.loc[valid_p.index, 'p_value_adjusted'] = p_adj
            ancova_df['Significant'] = ancova_df['p_value_adjusted'] < alpha
            
            # Reorder columns for side-by-side presentation
            cols = ['Variable', 'Effect', 'N', 'num_df', 'den_df', 'F_statistic', 'p_value_raw', 'p_value_adjusted', 'Significant', 'Partial_Eta_Squared']
            ancova_df = ancova_df[[c for c in cols if c in ancova_df.columns]]

    # Post-Hoc Table Processing
    posthoc_df = pd.concat(posthoc_results, ignore_index=True) if posthoc_results else pd.DataFrame()
    if not posthoc_df.empty:
        posthoc_df = posthoc_df.rename(columns={
            'p.value': 'p_value_raw',
            'p.value_adj': f'p_value_{adjust_method}'
        })
        posthoc_df['Significant'] = posthoc_df[f'p_value_{adjust_method}'] < alpha
        
        # Reorder columns for side-by-side presentation
        front_cols = ['Variable', 'Comparison_Type', 'contrast', 'Group', 'TimePoint', 'estimate', 'SE', 'df', 't.ratio', 'p_value_raw', 'p_value_adjusted', 'Significant']
        other_cols = [c for c in posthoc_df.columns if c not in front_cols]
        posthoc_df = posthoc_df[[c for c in front_cols if c in posthoc_df.columns] + other_cols]

    return ancova_df, posthoc_df

#3B. EV CORRELATIONS
def get_significance_stars(p):
    """Return significance stars based on p-value."""
    if p < 0.0001:
        return '****'
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'
    
def get_figure_2E_correlations(
    df_long, 
    col_x="Median Value (nm)", 
    col_y="Concentration/ml", 
    sex_col="sex", 
    time_col="time",
    stratify_by_sex=True,
    plot=True,
    figsize=(6.5, 5.2)
):
    time_order = ["baseline", "3min", "1hr", "2hrs"]
    time_labels = ["Time: Baseline", "Time: 3 min", "Time: 1 hour", "Time: 2 hours"]
    
    df = df_long.copy()
    df[col_x] = pd.to_numeric(df[col_x], errors='coerce')
    df[col_y] = pd.to_numeric(df[col_y], errors='coerce')
    df = df.dropna(subset=[col_x, col_y, time_col])
    
    overall_rows = []
    by_sex_rows = []

    # 1. CALCULATE STATS ACROSS TIME POINTS
    for t_val in time_order:
        sub_time = df[df[time_col] == t_val]
        
        # Overall Stats
        if len(sub_time) >= 3:
            r_o, p_o = pearsonr(sub_time[col_x], sub_time[col_y])
            z_o = np.arctanh(r_o)
            se_o = 1 / np.sqrt(len(sub_time) - 3)
            overall_rows.append({
                'TimePoint': t_val, 'Variable_X': col_x, 'Variable_Y': col_y,
                'N': len(sub_time), 'r': r_o, 'p_value_raw': p_o,
                'CI_95_lower': np.tanh(z_o - 1.96 * se_o),
                'CI_95_upper': np.tanh(z_o + 1.96 * se_o)
            })

        # Stratified by Sex Stats
        for sex_val in ['Female', 'Male']:
            sub_sex = sub_time[sub_time[sex_col].str.title() == sex_val]
            if len(sub_sex) >= 3:
                r_s, p_s = pearsonr(sub_sex[col_x], sub_sex[col_y])
                z_s = np.arctanh(r_s)
                se_s = 1 / np.sqrt(len(sub_sex) - 3)
                by_sex_rows.append({
                    'TimePoint': t_val, 'Sex': sex_val, 'Variable_X': col_x, 'Variable_Y': col_y,
                    'N': len(sub_sex), 'r': r_s, 'p_value_raw': p_s,
                    'CI_95_lower': np.tanh(z_s - 1.96 * se_s),
                    'CI_95_upper': np.tanh(z_s + 1.96 * se_s)
                })

    # Prepare DataFrames & Apply FDR Adjustments
    df_overall = pd.DataFrame(overall_rows)
    if not df_overall.empty:
        _, p_adj, _, _ = multipletests(df_overall['p_value_raw'], method='fdr_bh')
        df_overall['p_value_FDR'] = p_adj

    df_sex = pd.DataFrame(by_sex_rows)
    if not df_sex.empty:
        _, p_adj_s, _, _ = multipletests(df_sex['p_value_raw'], method='fdr_bh')
        df_sex['p_value_FDR'] = p_adj_s

    # 2. RENDER PLOT (IF PLOT=TRUE)
    if plot:
        xlim = (df[col_x].min() * 0.95, df[col_x].max() * 1.05)
        ylim = (df[col_y].min() * 0.95, df[col_y].max() * 1.05)
        fig, axes = plt.subplots(2, 2, figsize=figsize, constrained_layout=True)
        axes = axes.ravel()

        for idx, (t_val, tlab) in enumerate(zip(time_order, time_labels)):
            sub_time = df[df[time_col] == t_val]
            ax = axes[idx]
            custom_handles, custom_labels = [], []

            if stratify_by_sex:
                for s_val, color, marker in zip(['Female', 'Male'], ['0.6', '0.1'], ['s', 'o']):
                    subset = sub_time[sub_time[sex_col].str.title() == s_val]
                    if len(subset) >= 3:
                        row = df_sex[(df_sex['TimePoint'] == t_val) & (df_sex['Sex'] == s_val)].iloc[0]
                        label = f"{s_val}{get_significance_stars(row['p_value_raw'])}: r={row['r']:.2f} [{row['CI_95_lower']:.2f}, {row['CI_95_upper']:.2f}]"
                        sns.regplot(data=subset, x=col_x, y=col_y, ax=ax, color=color, scatter=False, ci=95)
                        sns.scatterplot(data=subset, x=col_x, y=col_y, ax=ax, color=color, marker=marker, s=40, legend=False)
                        custom_handles.append(Line2D([0], [0], color=color, lw=1.5, marker=marker, markersize=5))
                        custom_labels.append(label)
            else:
                if len(sub_time) >= 3:
                    row = df_overall[df_overall['TimePoint'] == t_val].iloc[0]
                    label = f"Overall{get_significance_stars(row['p_value_raw'])}: r={row['r']:.2f} [{row['CI_95_lower']:.2f}, {row['CI_95_upper']:.2f}]"
                    sns.regplot(data=sub_time, x=col_x, y=col_y, ax=ax, color='#0056b3', scatter=False, ci=95)
                    sns.scatterplot(data=sub_time, x=col_x, y=col_y, ax=ax, color='#0056b3', marker='o', s=45, legend=False)
                    custom_handles.append(Line2D([0], [0], color='#0056b3', lw=1.8, marker='o', markersize=6))
                    custom_labels.append(label)

            ax.set_title(tlab, fontweight='bold', fontsize=8.5)
            ax.set_xlim(xlim); ax.set_ylim(ylim)
            ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
            ax.grid(True, linestyle='--', alpha=0.3)
            if custom_handles:
                ax.legend(handles=custom_handles, labels=custom_labels, frameon=False, prop={'weight': 'bold', 'size': 7}, loc='best')

        fig.supxlabel("Median Size (nm)", fontweight='bold', fontsize=8.5)
        fig.supylabel("EV Concentration (/ml)", fontweight='bold', fontsize=8.5)
        plt.suptitle("Figure 2E: Correlation (EV Concentration vs Median Size)", y=1.02, fontweight='bold', fontsize=9.5)
        plt.show()

        # 3. DISPLAY SUMMARY TABLE BENEATH THE PLOT
        disp_df = df_sex if stratify_by_sex else df_overall
        disp_df_formatted = disp_df.copy()
        disp_df_formatted['r'] = disp_df_formatted['r'].round(3)
        disp_df_formatted['CI [95%]'] = disp_df_formatted.apply(lambda r: f"[{r['CI_95_lower']:.2f}, {r['CI_95_upper']:.2f}]", axis=1)
        disp_df_formatted['p_value_raw'] = disp_df_formatted['p_value_raw'].apply(lambda p: f"{p:.4f}" if p >= 0.0001 else "< 0.0001")
        disp_df_formatted['p_value_FDR'] = disp_df_formatted['p_value_FDR'].apply(lambda p: f"{p:.4f}" if p >= 0.0001 else "< 0.0001")
        
        cols_to_show = (['TimePoint', 'Sex', 'N', 'r', 'CI [95%]', 'p_value_raw', 'p_value_FDR'] 
                        if stratify_by_sex else 
                        ['TimePoint', 'N', 'r', 'CI [95%]', 'p_value_raw', 'p_value_FDR'])
        
        display(HTML(f"""
        <div style="margin-top: 10px; font-size: 11px;">
            <b>📊 Statistical Summary: Pearson Correlation (r) & FDR Corrections</b>
            {disp_df_formatted[cols_to_show].to_html(index=False, classes='table table-striped table-hover')}
        </div>
        """))

    return df_overall, df_sex

# --- 4. DASHBOARD RENDERER & WIDGETS ---
view_select = widgets.Dropdown(
    options=[
        '📄 Primary Manuscript Report (Jamovi PDF Output)', 
        'Figure 2A: EV Concentration Trajectory', 
        'Figure 2B: EV Size Trajectory', 
        'Figure 2CD: EMM Group Differences', 
        'Figure 2E: EV Conc vs Size Correlation (Overall)',
        'Figure 2E: EV Conc vs Size Correlation (By Sex)',      
        'ANCOVA Table (DF, F-stat, p-FDR, pes)',
        'Post-Hoc Pairwise Comparisons (Within & Between)'
    ],
    value='📄 Primary Manuscript Report (Jamovi PDF Output)', #default
    description='Display View:',
    style={'description_width': 'initial'}
)

export_btn = widgets.Button(
    description='Download All Model Stats & Data (.xlsx)',
    button_style='success',
    icon='download'
)

out = widgets.Output()

def render_dashboard(change=None):
    with out:
        out.clear_output(wait=True)
        selected_view = view_select.value
        
        mpl.rcParams['svg.fonttype'] = 'none'
        plt.rcParams.update({
            'font.family': 'sans-serif', 
            'font.sans-serif': ['Arial', 'Helvetica', 'FreeSans', 'DejaVu Sans', 'sans-serif'],
            'font.size': 8, 'font.weight': 'bold',
            'axes.titlesize': 8.5, 'axes.titleweight': 'bold',
            'axes.labelsize': 8, 'axes.labelweight': 'bold',
            'xtick.labelsize': 7, 'ytick.labelsize': 7,
            'axes.linewidth': 1.0, 'lines.linewidth': 1.2
        })
        
        sex_colors = {'Male': '0.2', 'Female': '0.6'}
        sex_markers = {'Male': 'o', 'Female': 's'}
        time_order = ["baseline", "3min", "1hr", "2hrs"]
        
        # --- VIEW 1: EMBEDDED JAMOVI PDF REPORT ---
        if selected_view == '📄 Primary Manuscript Report (Jamovi PDF Output)':
            # Raw GitHub PDF URL
            pdf_url = "https://raw.githubusercontent.com/ernestonifade/GLYMREG-Extracellular-Vesicle-Study/main/data/Jamovi_Statistical_Report_Figure2.pdf"
            
            # Mozilla PDF.js CDN viewer link
            pdfjs_viewer_url = f"https://mozilla.github.io/pdf.js/web/viewer.html?file={pdf_url}"

            display(HTML(f"""
            <div style="background-color: #e3f2fd; border-left: 5px solid #2196f3; padding: 12px 16px; margin-bottom: 15px; border-radius: 4px; font-size: 12px; line-height: 1.6; color: #0d47a1;">
                <b>📄 Primary Manuscript Statistical Report (Jamovi v2.3+):</b><br>
                This viewer displays the primary statistical export generated in Jamovi, which was used for reporting EV concentration/size p-values and annotated figure brackets in the manuscript. 
                <i>To explore the automated batch-processing model used for high-dimensional protein/cytokine screening, select any of the interactive views from the dropdown menu above.</i>
                <br><br>
                <a href="{pdf_url}" target="_blank" style="background-color: #2196f3; color: white; padding: 6px 12px; border-radius: 4px; text-decoration: none; font-weight: bold; display: inline-block;">
                    📥 Open / Download Full Jamovi PDF in New Tab ↗
                </a>
            </div>
            
            <!-- Reliable PDF.js Inline Viewer -->
            <iframe src="{pdfjs_viewer_url}" 
                    width="100%" 
                    height="750px" 
                    style="border: 1px solid #cccccc; border-radius: 4px; margin-top: 5px;">
            </iframe>
            """))
        
        elif selected_view == 'Figure 2A: EV Concentration Trajectory':
            fig, ax = plt.subplots(figsize=(4.5, 3.5), layout='constrained')
            var = "Concentration/ml"
            summary = (long_df.groupby(["time", "sex"])[var].agg(mean="mean", std="std", count="count").reset_index())
            summary["sem"] = summary["std"] / np.sqrt(summary["count"].clip(lower=1))
            
            for sex_val, group in summary.groupby("sex"):
                group = group.set_index("time").reindex(time_order).reset_index()
                ax.errorbar(
                    np.arange(len(time_order)), group["mean"], yerr=1.96*group["sem"],
                    fmt=f"{sex_markers.get(sex_val, 'o')}-", capsize=4, label=sex_val,
                    color=sex_colors.get(sex_val, 'black'), linewidth=1.5, markersize=6
                )
            ax.set_xticks(np.arange(len(time_order)))
            ax.set_xticklabels(['Baseline', '3 min', '1 hour', '2 hours'])
            ax.set_xlabel("Time Point"); ax.set_ylabel("EV Concentration (/ml) (± 95% CI)")
            ax.set_title("Temporal Dynamics: EV Concentration")
            
            summary['ci_upper'] = summary['mean'] + (1.96 * summary['sem'])  
            
            # Set bracket above the highest CI
            y_max = summary['ci_upper'].max() * 1.15  
            add_bracket(ax, 0, 3, y_max, y_max*0.03, "ns (p = 0.309)")
            ax.set_ylim(0, summary['ci_upper'].max() * 1.30)  
            
            formatter = mticker.ScalarFormatter(useMathText=True)
            formatter.set_scientific(True); formatter.set_powerlimits((-1, 1))
            ax.yaxis.set_major_formatter(formatter)
            ax.legend(frameon=False, prop={'weight': 'bold', 'size': 7})
            ax.grid(True, alpha=0.2, linestyle='--')
            plt.show()
            
        elif selected_view == 'Figure 2B: EV Size Trajectory':
            fig, ax = plt.subplots(figsize=(4.5, 3.5), layout='constrained')
            var = "Median Value (nm)"
            summary = (long_df.groupby(["time", "sex"])[var].agg(mean="mean", std="std", count="count").reset_index())
            summary["sem"] = summary["std"] / np.sqrt(summary["count"].clip(lower=1))
            
            for sex_val, group in summary.groupby("sex"):
                group = group.set_index("time").reindex(time_order).reset_index()
                ax.errorbar(
                    np.arange(len(time_order)), group["mean"], yerr=1.96*group["sem"],
                    fmt=f"{sex_markers.get(sex_val, 'o')}-", capsize=4, label=sex_val,
                    color=sex_colors.get(sex_val, 'black'), linewidth=1.5, markersize=6
                )
            ax.set_xticks(np.arange(len(time_order)))
            ax.set_xticklabels(['Baseline', '3 min', '1 hour', '2 hours'])
            ax.set_xlabel("Time Point"); ax.set_ylabel("Median Size (nm) (± 95% CI)")
            ax.set_title("Temporal Dynamics: EV Size")
            
            y_max = summary['mean'].max() * 1.12
            add_bracket(ax, 2, 3, y_max, y_max*0.02, "* p = 0.048")
            y_max = summary['mean'].max() * 1.20
            add_bracket(ax, 1, 2, y_max, y_max*0.02, "* p = 0.029")
            
            ax.legend(frameon=False, prop={'weight': 'bold', 'size': 7})
            ax.grid(True, alpha=0.2, linestyle='--')
            plt.show()

        elif selected_view == 'Figure 2CD: EMM Group Differences':
            fig, axes = plt.subplots(1, 2, figsize=(3.5, 3.5), layout='constrained')
            df_emm = pd.DataFrame({
                'Metabolite': ['EV Concentration (/ml)'] * 2 + ['EV Size (nm)'] * 2,
                'Group': ['Male', 'Female'] * 2,
                'EMM': [2.56e11, 1.94e11, 102.0, 120.0],
                'CI_lower': [1.44e11, 8.17e10, 93.7, 111.1],
                'CI_upper': [3.68e11, 3.06e11, 111.0, 129.0]
            })
            
            sub_size = df_emm[df_emm['Metabolite'] == 'EV Size (nm)']
            for idx, row in sub_size.iterrows():
                x_pos = 0 if row['Group'] == 'Male' else 1
                axes[0].errorbar(x_pos, row['EMM'], yerr=[[row['EMM'] - row['CI_lower']], [row['CI_upper'] - row['EMM']]],
                             fmt='o' if x_pos==0 else 's', color='0.2' if x_pos==0 else '0.6', capsize=5, markersize=7)
            axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Male', 'Female'])
            axes[0].set_ylabel('Adjusted EMM (nm)'); axes[0].set_title('EV Size')
            add_bracket(axes[0], 0, 1, 131, 2, "** p = 0.009")
            axes[0].grid(axis='y', linestyle='--', alpha=0.5)

            sub_conc = df_emm[df_emm['Metabolite'] == 'EV Concentration (/ml)']
            for idx, row in sub_conc.iterrows():
                x_pos = 0 if row['Group'] == 'Male' else 1
                axes[1].errorbar(x_pos, row['EMM'], yerr=[[row['EMM'] - row['CI_lower']], [row['CI_upper'] - row['EMM']]],
                             fmt='o' if x_pos==0 else 's', color='0.2' if x_pos==0 else '0.6', capsize=5, markersize=7)
            axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['Male', 'Female'])
            axes[1].set_ylabel('Adjusted EMM (/ml)'); axes[1].set_title('EV Concentration')
            add_bracket(axes[1], 0, 1, 3.9e11, 1e10, "p = 0.419 (ns)")
            
            formatter = mticker.ScalarFormatter(useMathText=True)
            formatter.set_scientific(True); formatter.set_powerlimits((-1, 1))
            axes[1].yaxis.set_major_formatter(formatter)
            axes[1].grid(axis='y', linestyle='--', alpha=0.5)
            plt.suptitle("Estimated Marginal Means (Baseline Adjusted)", fontweight='bold')
            plt.show()

        elif selected_view == 'ANCOVA Table (DF, F-stat, p-FDR, pes)':
            # Top Banner with JavaScript Scroll Button (No New Tabs)
            display(HTML("""
            <div style="background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 10px 14px; margin-bottom: 12px; border-radius: 4px; font-size: 12px; line-height: 1.5; color: #856404;">
                <b>📄 Reviewer Note on Output Alignment:</b> The table below uses an automated <code>R (afex/emmeans)</code> engine. 
                To inspect the original primary <b>Jamovi statistical report</b> used for manuscript figures: 
                <a href="https://github.com/ernestonifade/GLYMREG-Extracellular-Vesicle-Study/raw/main/Jamovi_Statistical_Report_Figure2.pdf" target="_blank" style="color: #533f03; font-weight: bold; text-decoration: underline;">
                    Download Jamovi PDF (GitHub) ↗
                </a>
            </div>
            <div style="background-color: #e9ecef; border-left: 4px solid #495057; padding: 8px 12px; margin-bottom: 12px; font-size: 12px; border-radius: 4px;">
                <b>ℹ️ Model Summary Guide:</b> Type-III RM-ANCOVA (baseline-adjusted). 
                <button onclick="document.getElementById('ancova-legend').scrollIntoView({behavior: 'smooth'});" 
                        style="background:none; border:none; color:#0056b3; text-decoration:underline; cursor:pointer; font-weight:bold; font-size:12px; padding:0;">
                    Jump to full table legend & column definitions ↓
                </button>
            </div>
            <h3>RM-ANCOVA Model Summary (Main & Interaction Effects)</h3>
            """))
            
            ancova_df, _ = run_full_ancova_with_posthoc(long_df)
            display(ancova_df)
            
            # Bottom Legend
            ancova_note = """
            <div id="ancova-legend" style="background-color: #f8f9fa; border-left: 4px solid #007bff; padding: 14px; margin-top: 15px; border-radius: 4px; font-size: 12px; line-height: 1.6; color: #212529;">
                <b>📊 Notes on ANCOVA Model Terms & Column Definitions:</b><br>
                • <b>Model Effect Definitions:</b>
                  <ul style="margin: 3px 0 8px 18px; padding: 0;">
                    <li><b>Time Effect (Main Effect):</b> Evaluates overall changes across recovery time points (3min, 1hr, 2hrs) pooling across sexes. Answers: <i>Does the marker change over time regardless of sex?</i></li>
                    <li><b>Group Effect / Sex (Main Effect):</b> Evaluates overall differences between Males and Females across all post-exercise time points combined. Answers: <i>Are overall marker levels higher in one sex regardless of time?</i></li>
                    <li><b>Time × Group Effect (Interaction):</b> Evaluates whether the temporal trajectory across recovery differs between Males and Females. Answers: <i>Does the sex difference depend on the time point, or do males and females respond differently over time?</i></li>
                  </ul>
                • <b>N:</b> Number of independent subjects included in the baseline-adjusted model.<br>
                • <b>num_df / den_df:</b> Numerator and denominator degrees of freedom (F<sub>num_df, den_df</sub>). Greenhouse-Geisser (GG) corrections applied for sphericity violations.<br>
                • <b>F_statistic:</b> Test statistic for the specified model effect.<br>
                • <b>p_value_raw:</b> Unadjusted p-value from Type-III ANOVA (<code>afex::aov_ez</code>).<br>
                • <b>p_value_adjusted:</b> False Discovery Rate adjusted p-value across primary outcomes (Benjamini-Hochberg procedure).<br>
                • <b>Partial_Eta_Squared (η<sub>p</sub>²):</b> Effect size (Small ≈ 0.01, Medium ≈ 0.06, Large ≥ 0.14).<br>
                • <b>NaN Values:</b> Not Applicable for specific single-comparison rows.
            </div>
            """
            display(HTML(ancova_note))

        elif selected_view == 'Post-Hoc Pairwise Comparisons (Within & Between)':
            # Top Banner with JavaScript Scroll Button (No New Tabs)
            display(HTML("""
            <div style="background-color: #e9ecef; border-left: 4px solid #495057; padding: 8px 12px; margin-bottom: 12px; font-size: 12px; border-radius: 4px;">
                <b>ℹ️ Contrasts Guide:</b> Baseline-adjusted EMMs. 
                <button onclick="document.getElementById('posthoc-legend').scrollIntoView({behavior: 'smooth'});" 
                        style="background:none; border:none; color:#0056b3; text-decoration:underline; cursor:pointer; font-weight:bold; font-size:12px; padding:0;">
                    Jump to full table legend & NaN explanation ↓
                </button>
            </div>
            <h3>Post-Hoc Pairwise Contrasts (emmeans)</h3>
            """))
            
            _, posthoc_df = run_full_ancova_with_posthoc(long_df)
            display(posthoc_df)
            
            # Bottom Legend
            posthoc_note = """
            <div id="posthoc-legend" style="background-color: #f8f9fa; border-left: 4px solid #28a745; padding: 14px; margin-top: 15px; border-radius: 4px; font-size: 12px; line-height: 1.6; color: #212529;">
                <b>📊 Notes on Pairwise Contrasts & Column Layout:</b><br>
                • <b>Comparison_Type:</b> 
                  <ul style="margin: 2px 0 6px 18px; padding: 0;">
                    <li><i>Between-Group:</i> Male vs. Female comparisons at each post-exercise time point.</li>
                    <li><i>Within-Group:</i> Pairwise time point shifts across recovery, stratified by sex.</li>
                  </ul>
                • <b>estimate:</b> Difference in Baseline-Adjusted Estimated Marginal Means (EMMs). A negative value means the first condition is lower than the second.<br>
                • <b>SE & df:</b> Standard Error and degrees of freedom (df = 18.0) for the contrast.<br>
                • <b>t.ratio:</b> Test statistic (estimate / SE).<br>
                • <b>p_value_raw:</b> Unadjusted p-value from Type-III ANOVA (<code>afex::aov_ez</code>).<br>
                • <b>p_value_adjusted:</b> False Discovery Rate adjusted p-value across primary outcomes (Benjamini-Hochberg procedure).<br>
                • <b>Why NaN appears in columns:</b>
                  <ul style="margin: 2px 0 0 18px; padding: 0;">
                    <li><i>NaN in <b>Group</b>:</i> Between-Group contrasts evaluate Male vs. Female across groups simultaneously.</li>
                    <li><i>NaN in <b>TimePoint</b>:</i> Within-Group contrasts evaluate time trajectories within a single sex group across time points.</li>
                  </ul>
            </div>
            """
            display(HTML(posthoc_note))
            
        elif selected_view == 'Figure 2E: EV Conc vs Size Correlation (Overall)':
            display(HTML("""
            <div style="background-color: #f8f9fa; border-left: 4px solid #28a745; padding: 10px 14px; margin-bottom: 12px; border-radius: 4px; font-size: 11px; line-height: 1.5; color: #343a40;">
                <b>Figure 2E (Overall Pooled):</b> Pearson correlation (r) and 95% Confidence Intervals calculated across the combined cohort at each recovery time point regardless of sex.
            </div>
            """))
            get_figure_2E_correlations(long_df, stratify_by_sex=False, plot=True)
            
        elif selected_view == 'Figure 2E: EV Conc vs Size Correlation (By Sex)':
            display(HTML("""
            <div style="background-color: #f8f9fa; border-left: 4px solid #007bff; padding: 10px 14px; margin-bottom: 12px; border-radius: 4px; font-size: 11px; line-height: 1.5; color: #343a40;">
                <b>Figure 2E (Stratified by Sex):</b> Pearson correlation (r) and 95% Confidence Intervals calculated independently for Males and Females at each recovery time point.
            </div>
            """))
            get_figure_2E_correlations(long_df, stratify_by_sex=True, plot=True)

# --- INSTANT DIRECT DOWNLOAD HANDLER ---
def export_all_data(b):
    file_name = "Figure2_RM_ANCOVA_Full_Stats_Report.xlsx"
    ancova_df, posthoc_df = run_full_ancova_with_posthoc(long_df)
    corr_overall_df, corr_sex_df = get_figure_2d_correlations(long_df, plot=False)
    
    # Save multi-sheet Excel file containing BOTH ANCOVA and Post-Hoc tables
    with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
        ancova_df.to_excel(writer, sheet_name='RM_ANCOVA_Model_Stats', index=False)
        posthoc_df.to_excel(writer, sheet_name='PostHoc_Pairwise_Contrasts', index=False)
        corr_overall_df.to_excel(writer, sheet_name='Correlation_Overall', index=False)
        corr_sex_df.to_excel(writer, sheet_name='Correlation_By_Sex', index=False)
        long_df.to_excel(writer, sheet_name='Raw_EV_Data', index=False)
        
    # Encode file into Base64
    with open(file_name, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    # Trigger IMMEDIATE browser download via invisible iframe (No second click needed!)
    js_download = f"""
    <script>
        var link = document.createElement('a');
        link.href = 'data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}';
        link.download = '{file_name}';
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    </script>
    <div style="color: #28a745; font-weight: bold; font-size: 12px; margin-top: 5px;">
        ✅ Downloading <i>{file_name}</i> (contains ANCOVA Stats + Post-Hoc Contrasts + Raw Data)...
    </div>
    """
    with out:
        display(HTML(js_download))

view_select.observe(render_dashboard, names='value')
export_btn.on_click(export_all_data)

display(widgets.HBox([view_select, export_btn]))
display(out)
render_dashboard()